# A3b -- the vicinity pseudo-granule control

Answers **Reviewer #2's literal round-1 suggestion**, re-requested in round 2 and never run:

> "Or alternatively, define **pseudo-granules in the direct vicinity of actual granules** as a
> negative control."
>
> — and in round 2: *"A direct check at the detection step, such as the pseudo-granule negative
> control I suggested, would settle the question."*

### Why the vicinity, specifically

This is the sharpest available answer to *structured* ambient. A sphere placed a few um away
shares the granule's local environment -- the same cell density, the same plaque proximity, the
same brain region, the same z-plane -- but is not a detected aggregate. If granules were merely
locally-elevated ambient, the offset spheres would look the same. A tissue-wide random location
would not test that, because it also changes the neighbourhood.

### Three design decisions that decide whether this survives scrutiny

**Offsets are in-plane.** `layer_z` takes only **7 discrete values** (0, 1.5 ... 9.0 um), and both
`profile()` and `nc_filter()` query at `layer_z`. A 3D direction would push the centre off the grid
and break comparability with every real granule.

**The transcript-count comparison is a tautology, so it does not carry the argument.** `sphere_r`
is the *minimum enclosing radius* of the DBSCAN core points (`miniball.get_bounding_ball` on the
deduplicated cluster). It is an order statistic: the sphere is maximally dense by construction, so
any displaced copy at the same radius must capture no more. Reporting "pseudo-granules capture
fewer transcripts" and stopping would be reporting an algebraic identity. It is kept as
description; the decision rests on section 4.

**The detection predicate is eps-connectivity, not a count.** Three transcripts scattered across a
4 um sphere are not `eps = 1.5`-connected, so "contains >= 3 transcripts" badly overstates how
detectable a pseudo-granule is. Section 4 applies DBSCAN's own core-point definition on the
**same seed gene** -- does any seed-gene transcript inside the sphere have `>= min_samples`
neighbours within `eps` in that gene's full point cloud -- so it answers "would the detector have
seeded a cluster here" exactly, with no clustering fit. "Any of 20 markers" is roughly 20x easier
than "3 Camk2a". Note this is the **seeding step only**: a failing pseudo-granule is never detected
at all rather than detected-then-filtered, and a passing one would still face the size, in-soma and
NC filters, so the reported rate is an upper bound.

### On rejecting offsets that land on a real granule

Measured before deciding: per-plane 2D granule coverage is only **1.9% (WT) / 1.5% (AD)**
(`sum(pi r^2) / (tissue_area x 7 planes)`), so rejecting granule-overlapping offsets discards few
candidates and does **not** meaningfully bias the sample toward granule-sparse space. (A 3D
nearest-neighbour distance of 2.68 um looks alarming but is misleading -- it counts neighbours on
adjacent z-planes.) Both arms are reported anyway, and in the unrejected arm the fraction of
offsets that land on a real granule is itself a **result**: it measures how much of the immediate
vicinity is already called.

### Where this sits in the run

Needs `set1_*/sphere_dict.parquet` from the HGCC array (the seed gene section 4 matches on). Section 6 needs nothing from HGCC. Runs **once, top to bottom, with nothing to adjust** -- see the runbook in
`README.md`.

**Run this notebook from `R2_revision/ambient_controls/`, on the `mcDETECT-env` kernel.**

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
import zlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.spatial import cKDTree

sys.path.insert(0, str(Path.cwd()))          # run this notebook from ambient_controls/
import a3_config as C
import a3_common as A3

warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

# -------------------- runtime gates -------------------- #
# THE DEFAULTS BELOW PRODUCE THE FINAL TABLES. Run top to bottom, once, and change nothing.
DRY_RUN = False          # True -> subsample for a cheap smoke pass over every cell. The tables a
                         #   dry run writes are NOT final; set it back to False and rerun.
VALIDATE = True          # section 7 correctness gates. On by default (A1/A2 have them off): they
                         #   are the LAST section, so every table is already written before they
                         #   run.
RUN_PREDICATE = True     # section 4 -- the load-bearing statistic
RUN_ROUGH_VARIANT = True # section 6 -- the zero-placement-bias variant, needs no HPC output
SEED = C.VICINITY_SEED

MAX_GRANULES = 200_000 if DRY_RUN else None   # None = all 681K/399K. Section 4's predicate is the
                                              # slow step, and it is what the whole control rests
                                              # on, so the final run must see every granule.

C.ensure_dirs()
OUT = C.A3B_DIR

print("writing to :", OUT)
print("arms       :", list(C.VICINITY_ARMS))
print("offsets    :", C.VICINITY_D, "um absolute +", C.VICINITY_D_RELATIVE, "x sphere_r")
print("predicate  : DBSCAN core point of the SEED gene inside the sphere"
      f" | eps = {C.EPS} | min_samples = {C.DETECT_KWARGS_FINE['minspl']}")
print("stratify on:", C.VICINITY_MATCH_ON, "(reported within, NOT enforced at placement)")

writing to : /Users/chenyang/Desktop/mcDETECT/R2_revision/ambient_controls/output/a3b
arms       : ['unrejected', 'rejected']
offsets    : [5.0, 10.0, 20.0, 50.0] um absolute + [2.0, 3.0] x sphere_r
predicate  : DBSCAN core point of the SEED gene inside the sphere | eps = 1.5 | min_samples = 3
stratify on: ['brain_area', 'density_quintile'] (reported within, NOT enforced at placement)


/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-p

## 1. Source granules, tissue mask and density strata

The source set is the published Set 2. Three things are attached to each granule before any
offset is placed. The first two are **stratification keys, not placement constraints** --
`place_vicinity_spheres` draws a uniform-random in-plane angle and rejects out-of-tissue and
in-nucleus placements (the tissue-wide random floor in section 4 applies the identical rule,
so the floor and the arms are built the same way and the elevation ratio compares like with
like),
in-nucleus and (rejected arm) granule-overlapping offsets. The pseudo-granule *inherits* the
source's labels, because a <= 50 um displacement rarely leaves a 25 um density cell or a brain
region, and every result is then reported **within** each level. Describe this as a stratified
comparison, never as a matched design.

* **`brain_area`** -- already on `granules.parquet` (nearest-spot label, `3_detection.py:88-95`).
* **local transcript-density quintile** on a 25 um lattice. This is the direct answer to "ambient
  is denser where cells are denser, or near plaques": source and copy sit in the same density
  stratum by construction, so a difference measured *within* a stratum cannot be a density
  difference.
* **`seed_gene`** from the persisted `sphere_dict`. `granules.parquet["gene"]` is stale for merged
  granules -- `_remove_overlaps` updates only `sphere_x/y/z`, `layer_z` and `sphere_r` -- so the
  recorded gene cannot be trusted for the seed-matched predicate. Where the `sphere_dict` is not
  yet on disk, the recorded gene is used and the run is recorded in `source_summary.csv`.

In [2]:
sources, masks, dens, nuc = {}, {}, {}, {}

for sample in C.SAMPLES:
    g = pd.read_parquet(C.mcdetect_granules_path(sample))
    if MAX_GRANULES and len(g) > MAX_GRANULES:
        g = g.sample(MAX_GRANULES, random_state=SEED).reset_index(drop=True)
    tx = A3.load_transcripts(sample, columns=["global_x", "global_y", "global_z", "target",
                                              "overlaps_nucleus"])

    mask, xb, yb = A3.tissue_mask(tx)
    qgrid, qxb, qyb = A3.density_quintiles(tx)

    # Nucleus point cloud, so offsets landing inside nuclei can actually be rejected -- both
    # arms are documented to do this and previously neither did.
    nuc_tx = tx[tx["overlaps_nucleus"] == 1]
    nuc[sample] = cKDTree(nuc_tx[["global_x", "global_y", "global_z"]].to_numpy(float))

    qi = np.clip(np.searchsorted(qxb, g["sphere_x"], side="right") - 1, 0, qgrid.shape[0] - 1)
    qj = np.clip(np.searchsorted(qyb, g["sphere_y"], side="right") - 1, 0, qgrid.shape[1] - 1)
    g["density_quintile"] = qgrid[qi, qj]

    # seed gene: prefer the persisted pre-merge dict; fall back to the stale column and say so
    sd_path = C.sphere_dict_path("set1", sample)
    if sd_path.exists():
        sd = pd.read_parquet(sd_path)
        tree = cKDTree(sd[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float))
        dnn, nn = tree.query(g[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float), k=1)
        g["seed_gene"] = sd["seed_gene"].to_numpy()[nn]
        g["seed_gene_d"] = dnn
        # The join is exact for unmerged granules; for MERGED ones the centre was refit by
        # miniball over the union, so the nearest pre-merge sphere is a guess. Cap it at the
        # granule's own radius and record the match rate rather than letting a distant sphere
        # silently donate its gene.
        bad = dnn > g["sphere_r"].to_numpy()
        g.loc[bad, "seed_gene"] = pd.NA
        g["seed_gene_source"] = "sphere_dict"
        print(f"[{sample}] seed gene matched for {(~bad).mean():.1%} of granules "
              f"(median NN distance {np.median(dnn):.3f} um)")
    else:
        g["seed_gene"] = g["gene"]
        g["seed_gene_d"] = np.nan
        g["seed_gene_source"] = "granules.parquet (STALE after merge_sphere)"
        print(f"[{sample}] WARNING: sphere_dict missing -- seed gene falls back to the stale column")

    g = g[g["seed_gene"].notna()].reset_index(drop=True)
    sources[sample] = g
    masks[sample] = (mask, xb, yb)
    dens[sample] = (qgrid, qxb, qyb)
    print(f"[{sample}] {len(g):,} source granules | "
          f"density strata {sorted(g['density_quintile'].unique())}")
    del tx

pd.DataFrame([dict(sample=s, n=len(v),
                   seed_gene_source=v["seed_gene_source"].iloc[0],
                   median_seed_gene_d=float(np.nanmedian(v["seed_gene_d"])))
              for s, v in sources.items()]).to_csv(OUT / "source_summary.csv", index=False)

[WT] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_WT_1/processed_data/transcripts.parquet


[WT] 103,398,068 transcripts


[WT] seed gene matched for 100.0% of granules (median NN distance 0.000 um)
[WT] 681,273 source granules | density strata [0, 1, 2, 3, 4]


[AD] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_AD_1/processed_data/transcripts.parquet


[AD] 68,876,647 transcripts


[AD] seed gene matched for 100.0% of granules (median NN distance 0.000 um)
[AD] 398,766 source granules | density strata [0, 1, 2, 3, 4]


## 2. Placement -- two arms, an offset sweep

`unrejected` rejects only in-nucleus and out-of-tissue offsets; `rejected` additionally rejects
offsets overlapping a real granule, using mcDETECT's **own** merge predicate and nothing stricter
(a pseudo-granule that merely intersects a real granule is no more inadmissible than two real
granules that intersect -- which they routinely do, since the merge rule requires centres within
`0.4*r`).

The offset sweep spans `2r`, `3r`, 5, 10, 20 and 50 um. The deliverable in section 5 is the
**shape** of the curve, not any single distance.

In [3]:
pseudo_frames, place_status = [], []

for sample in C.SAMPLES:
    g = sources[sample]
    mask, xb, yb = masks[sample]
    # 3D, matching mcDETECT's merge predicate. A 2D tree rejects an offset for overlapping a
    # granule on ANY z-plane -- stricter than the rule the arm claims, and stricter than real
    # granules obey.
    real_tree = cKDTree(g[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float))
    real_r = g["sphere_r"].to_numpy(float)

    offsets = ([("abs", d, np.full(len(g), float(d))) for d in C.VICINITY_D] +
               [("rel", m, m * g["sphere_r"].to_numpy(float)) for m in C.VICINITY_D_RELATIVE])

    for kind, label, dvec in offsets:
        for arm in C.VICINITY_ARMS:
            # crc32, not hash(): Python's string hash is salted per process, so the placement
            # would not reproduce across runs.
            seed = zlib.crc32(f"{C.VICINITY_SEED}|{sample}|{kind}|{label}|{arm}".encode())
            rng = np.random.default_rng(seed)
            ps = A3.place_vicinity_spheres(g, mask, xb, yb, dvec, arm,
                                           real_tree=real_tree, real_r=real_r,
                                           nuc_tree=nuc[sample], rng=rng)
            ps["sample"], ps["d_kind"], ps["d_label"] = sample, kind, label
            pseudo_frames.append(ps)
            place_status.append(dict(sample=sample, arm=arm, d_kind=kind, d_label=label,
                                     n=len(ps), n_accepted=int(ps["accepted"].sum()),
                                     frac_accepted=float(ps["accepted"].mean()),
                                     mean_retry=float(ps["n_retry"].mean())))
            print(f"[{sample}] {arm:<11} d={kind}:{label:<5} "
                  f"accepted {ps['accepted'].mean():.3f}")

pseudo = pd.concat(pseudo_frames, ignore_index=True)
status = pd.DataFrame(place_status)
status.to_csv(OUT / "placement_status.csv", index=False)
display(status)

[WT] unrejected  d=abs:5.0   accepted 0.999


[WT] rejected    d=abs:5.0   accepted 0.999


[WT] unrejected  d=abs:10.0  accepted 0.999


[WT] rejected    d=abs:10.0  accepted 0.999


[WT] unrejected  d=abs:20.0  accepted 0.999


...

[AD] unrejected  d=rel:2.0   accepted 0.999


[AD] rejected    d=rel:2.0   accepted 0.999


[AD] unrejected  d=rel:3.0   accepted 0.999


[AD] rejected    d=rel:3.0   accepted 0.999


,sample,arm,d_kind,d_label,n,n_accepted,frac_accepted,mean_retry
0,WT,unrejected,abs,5.0,681273,680688,0.999141,0.389041
1,WT,rejected,abs,5.0,681273,680626,0.999050,0.419846
2,WT,unrejected,abs,10.0,681273,680677,0.999125,0.501194
3,WT,rejected,abs,10.0,681273,680681,0.999131,0.528136
4,WT,unrejected,abs,20.0,681273,680754,0.999238,0.544076
5,WT,rejected,abs,20.0,681273,680715,0.999181,0.570823
6,WT,unrejected,abs,50.0,681273,680969,0.999554,0.602590
7,WT,rejected,abs,50.0,681273,680893,0.999442,0.630129
8,WT,unrejected,rel,2.0,681273,680803,0.999310,0.214042
9,WT,rejected,rel,2.0,681273,680643,0.999075,0.246870


In [4]:
# How often does an offset land on a real granule? In the `unrejected` arm this is not a
# nuisance -- it is a RESULT: it measures how much of the immediate vicinity is already called,
# and it is the number that decides whether the `rejected` arm could have biased anything.
ov_rows = []
for (sample, kind, label), grp in pseudo[pseudo["arm"] == "unrejected"].groupby(
        ["sample", "d_kind", "d_label"], observed=True):
    acc = grp[grp["accepted"]]
    if len(acc) == 0:
        continue
    hit, cnt = A3.overlap_pairs(acc, sources[sample], criterion="intersect", z_col="layer_z")
    ov_rows.append(dict(sample=sample, d_kind=kind, d_label=label, n=len(acc),
                        frac_on_real_granule=float(hit.mean()),
                        mean_n_real_overlapped=float(cnt.mean())))
ov = pd.DataFrame(ov_rows)
ov.to_csv(OUT / "vicinity_overlap_with_real.csv", index=False)
display(ov)

,sample,d_kind,d_label,n,frac_on_real_granule,mean_n_real_overlapped
0,AD,abs,5.0,398381,0.328168,0.583108
1,AD,abs,10.0,398463,0.306761,0.530200
2,AD,abs,20.0,398558,0.300609,0.523111
3,AD,abs,50.0,398602,0.289976,0.498123
4,AD,rel,2.0,398441,0.672719,1.088603
5,AD,rel,3.0,398292,0.334564,0.571463
6,WT,abs,5.0,680688,0.242675,0.364653
7,WT,abs,10.0,680677,0.217594,0.313716
8,WT,abs,20.0,680754,0.208886,0.297372
9,WT,abs,50.0,680969,0.203664,0.286824


## 3. Profiling -- descriptive only

Real and pseudo spheres profiled by **identical** code (`sphere_features.profile_spheres`, a
vectorised re-implementation of `mcDETECT.model.profile` -- same ball, same `layer_z` centre, same
counting rule).

Read this section as description, not as the result. As stated in the header, a matched-radius
count comparison is an algebraic identity: `sphere_r` is the minimum enclosing radius of the exact
core points, so the real sphere is maximally dense by construction. What is *not* pre-ordained is
the **shape** of the gap -- how the in-nucleus ratio, NC ratio and unique-gene count differ -- and
those are worth reporting side by side as a filter funnel.

**The funnel's in-soma stage is meaningful for the real arm only.** Accepted pseudo-granules are
nucleus-free by construction (see section 2), so `n_extrasomatic` is reported over **non-empty**
spheres only, and `n_empty` is reported beside it as a result in its own right rather than being
silently folded into a filter count.


In [5]:
prof_summ, prof_hist, funnel_rows = [], [], []

for sample in C.SAMPLES:
    tx = A3.load_transcripts(sample)

    arms = [("real", sources[sample])]
    for (kind, label, arm), grp in pseudo[pseudo["sample"] == sample].groupby(
            ["d_kind", "d_label", "arm"], observed=True):
        acc = grp[grp["accepted"]].reset_index(drop=True)
        if len(acc):
            arms.append((f"pseudo|{arm}|{kind}:{label}", acc))

    for name, frame in arms:
        # Go through the a3_common WRAPPER, not sf.profile_spheres directly. The wrapper is what
        # resolves the 18-gene NC list (a3_config's provenance rule: a NEW population must not be
        # scored against a list containing Gria2, a granule marker) and what opts into
        # C.KG_BUFFER. Calling the delegate by hand silently took the published 19-gene list.
        feats, _X, _g = A3.profile_spheres(frame, sample=sample, transcripts=tx, verbose=False)
        for measure, col in [("n_total", "n_total"), ("n_marker", "n_marker"),
                             ("in_soma_ratio", "in_soma_ratio_all"),
                             ("nc_ratio", "nc_ratio_all")]:
            if col not in feats:
                continue
            s, h = A3.record_distribution(feats[col], measure, C.HIST_BINS[measure],
                                          sample=sample, arm=name)
            prof_summ.append(s)
            prof_hist.extend(h)
        # THE IN-SOMA STAGE IS VACUOUS FOR THE PSEUDO ARMS, AND MUST NOT BE REPORTED AS THOUGH
        # IT WERE NOT. place_vicinity_spheres rejects any offset whose sphere contains a
        # nucleus-overlapping transcript, so every accepted pseudo-granule has in_soma_ratio == 0
        # BY CONSTRUCTION; and _safe_div returns NaN for an EMPTY sphere, which fails `< thr`, so
        # a naive `(in_soma_ratio_all < IN_SOMA_THR).sum()` silently counts empty spheres as
        # "somatic". That is what the previous version did: its n_extrasomatic was exactly
        # n * (1 - frac_zero(n_total)), i.e. the EMPTY-sphere fraction wearing an in-soma label.
        #
        # Report the two things separately and honestly. n_empty is a real and useful result --
        # a displaced sphere often contains no transcripts at all -- and the in-soma count is
        # taken over NON-EMPTY spheres only, where the ratio is defined.
        nonempty = feats["n_total"].to_numpy() > 0
        in_soma = feats["in_soma_ratio_all"].to_numpy()
        nc = feats["nc_ratio_all"].to_numpy()
        keep_soma = nonempty & (in_soma < C.IN_SOMA_THR)
        funnel_rows.append(dict(
            sample=sample, arm=name, n=len(feats),
            n_empty=int((~nonempty).sum()),
            n_nonempty=int(nonempty.sum()),
            n_extrasomatic=int(keep_soma.sum()),
            n_nc_clean=int((keep_soma & (nc < C.NC_THR)).sum()),
        ))
    del tx

pd.DataFrame(prof_summ).to_csv(OUT / "profile_summary.csv", index=False)
pd.DataFrame(prof_hist).to_parquet(OUT / "profile_histogram.parquet", index=False)
fun = pd.DataFrame(funnel_rows)
fun.to_csv(OUT / "profile_funnel.csv", index=False)
display(fun)

[WT] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_WT_1/processed_data/transcripts.parquet


[WT] 103,398,068 transcripts


[AD] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_AD_1/processed_data/transcripts.parquet


[AD] 68,876,647 transcripts


,sample,arm,n,n_empty,n_nonempty,n_extrasomatic,n_nc_clean
0,WT,real,681273,0,681273,673540,673250
1,WT,pseudo|rejected|abs:5.0,680626,187563,493063,492202,479186
2,WT,pseudo|unrejected|abs:5.0,680688,185434,495254,494339,481444
3,WT,pseudo|rejected|abs:10.0,680681,206084,474597,473554,461278
4,WT,pseudo|unrejected|abs:10.0,680677,205098,475579,474638,462465
5,WT,pseudo|rejected|abs:20.0,680715,211450,469265,468226,455899
6,WT,pseudo|unrejected|abs:20.0,680754,209484,471270,470325,458220
7,WT,pseudo|rejected|abs:50.0,680893,214194,466699,465760,453408
8,WT,pseudo|unrejected|abs:50.0,680969,213534,467435,466477,454825
9,WT,pseudo|rejected|rel:2.0,680643,119781,560862,560327,542639


## 4. The detection predicate -- would the detector have fired here?

This is what the reviewer's control actually asks, and it is the only statistic in this notebook
that is not pre-ordained by the geometry.

For each accepted pseudo-granule: pull the **seed gene's** transcripts inside the sphere and ask
whether any of them is a `DBSCAN(eps = 1.5, min_samples = 3)` **core point** in that gene's full
point cloud. Seed-matched, because "any of the 20 markers" is roughly 20x easier than "3 Camk2a",
and Camk2a alone is 47% of the published granules.

Reported against two references: the same predicate evaluated at the **real** granule, and at
**tissue-wide random locations**, which is the floor the offset curve should approach as `d` grows.

### What this covers, and what it does not

**The predicate is the SEEDING step only.** A pseudo-granule that fails it is not "detected and
then filtered out" -- no core point means DBSCAN forms no cluster, no candidate sphere is ever
constructed, and it never reaches the size / in-soma / NC filters. A pseudo-granule that *passes*
has cleared seeding alone and would still have to survive the size, in-soma and NC filters.
`frac_detect` is therefore an **upper bound** on the fraction that would have become a published
granule. The two statistics are computed on the full accepted population independently and are
deliberately **not chained**.

**Do NOT quote section 3's in-soma stage as a further filter on these.** It cannot be one:
`place_vicinity_spheres` rejects any offset whose sphere touches a nucleus, so every accepted
pseudo-granule has `in_soma_ratio == 0` by construction. An earlier version of section 3 appeared
to show a further ~28% loss there; that number was the **empty-sphere** fraction leaking through a
`NaN < thr` comparison, not a somatic-overlap loss. Section 3 now reports `n_empty` and the
in-soma count separately, and the emptiness is itself a result worth stating -- a large minority of
displaced spheres contain no transcripts at all.

**The real arm now reaches ~100%, and the reason it previously did not was this notebook, not
mcDETECT.** Every published granule was produced by DBSCAN, so all of them must clear a
DBSCAN-based test. The old loss was in the containment query: `sphere_r` is the *minimum
enclosing* radius of the cluster that defined the granule, so its support points sit exactly **on**
the surface and `query_ball_point(centre, sphere_r)` dropped them to floating point. The evidence
was in the output itself -- `median_n_local = 2` for real granules, below the `min_samples = 3`
that must have been present when the cluster formed. The query now runs at
`sphere_r + C.KG_BUFFER`, the buffer this repository already documents as mandatory for any
recount of a cluster's own transcripts.

The correction is **not** symmetric between the arms, and the earlier claim that it was is wrong.
A pseudo-sphere has no support points on its surface, so the buffer moves it only by the generic
volume term (~3% at the median radius); the real arm was losing an entire boundary shell. The fix
therefore **widens** the real-vs-pseudo gap rather than flattering it. The `rel:2.0` arm deserves
its own look: it sits externally tangent to its source granule, so the buffer opens a small lens
around the tangent point where the source's own support points live -- check that arm against the
new numbers rather than assuming the old geometric argument survives.


In [6]:
if RUN_PREDICATE:
    pred_rows = []
    for col in ("would_detect", "gene_missing"):
        pseudo[col] = False
    pseudo["n_local"] = 0
    for sample in C.SAMPLES:
        tx = A3.load_transcripts(sample, columns=["global_x", "global_y", "global_z", "target"])
        by_gene = {}
        for g in C.SYN_GENES:
            sub = tx[tx["target"] == g]
            if len(sub) == 0:
                continue
            coords = sub[["global_x", "global_y", "global_z"]].to_numpy(float)
            by_gene[g] = (cKDTree(coords), coords)

        # real granules -- the ceiling
        real = sources[sample].copy()
        real["accepted"] = True
        r_pred = A3.dbscan_core_predicate(real, by_gene)
        pred_rows.append(dict(sample=sample, arm="real", d_kind="-", d_label="-",
                              n=len(r_pred), frac_detect=float(r_pred["would_detect"].mean()),
                              median_n_local=float(r_pred["n_local"].median())))

        # Tissue-wide random -- the floor / asymptote.
        #
        # THIS ARM MUST BE BUILT UNDER THE SAME RULES AS THE VICINITY ARMS, or the elevation
        # ratio quoted against it compares two different populations. place_vicinity_spheres
        # rejects offsets whose sphere touches a nucleus; this arm previously rejected only
        # out-of-tissue, and nuclei are transcript-dense, so the floor was inflated while the
        # arms were not. Apply the identical in-nucleus rejection here.
        mask, xb, yb = masks[sample]
        nuc_tree = nuc[sample]
        rng = np.random.default_rng(SEED)
        rnd = real.copy()
        r_all = rnd["sphere_r"].to_numpy(float)
        z_all = rnd["layer_z"].to_numpy(float)
        rx, ry, ok = np.zeros(len(rnd)), np.zeros(len(rnd)), np.zeros(len(rnd), bool)
        for _ in range(C.VICINITY_MAX_RETRY):
            todo = ~ok
            if not todo.any():
                break
            sel = np.flatnonzero(todo)
            cx = rng.uniform(xb[0], xb[-1], sel.size)
            cy = rng.uniform(yb[0], yb[-1], sel.size)
            good = A3.in_tissue(cx, cy, mask, xb, yb)
            if nuc_tree is not None and good.any():
                gi = np.flatnonzero(good)
                hit = np.asarray(nuc_tree.query_ball_point(
                    np.column_stack([cx[gi], cy[gi], z_all[sel[gi]]]), r_all[sel[gi]],
                    workers=-1, return_length=True), dtype=np.int64)
                good[gi[hit > 0]] = False
            idx = sel[good]
            rx[idx], ry[idx] = cx[good], cy[good]
            ok[idx] = True
        rnd["sphere_x"], rnd["sphere_y"], rnd["accepted"] = rx, ry, ok
        print(f"[{sample}] random floor placed {int(ok.sum()):,}/{len(rnd):,} "
              f"(in-tissue AND out-of-nucleus, same rule as the vicinity arms)", flush=True)
        q_pred = A3.dbscan_core_predicate(rnd, by_gene)
        pred_rows.append(dict(sample=sample, arm="random_tissue", d_kind="-", d_label="-",
                              n=int(ok.sum()), frac_detect=float(q_pred["would_detect"][ok].mean()),
                              median_n_local=float(q_pred["n_local"][ok].median())))

        # the vicinity arms -- computed ONCE here and written back onto `pseudo`, so section 5
        # can stratify without refitting (it previously recomputed the whole thing).
        m_s = (pseudo["sample"] == sample) & pseudo["accepted"]
        if m_s.any():
            pp = A3.dbscan_core_predicate(pseudo.loc[m_s], by_gene)
            pseudo.loc[m_s, "would_detect"] = pp["would_detect"].to_numpy()
            pseudo.loc[m_s, "n_local"] = pp["n_local"].to_numpy()
            pseudo.loc[m_s, "gene_missing"] = pp["gene_missing"].to_numpy()
        for (kind, label, arm), grp in pseudo[m_s].groupby(
                ["d_kind", "d_label", "arm"], observed=True):
            pred_rows.append(dict(sample=sample, arm=arm, d_kind=kind, d_label=label,
                                  n=len(grp), frac_detect=float(grp["would_detect"].mean()),
                                  median_n_local=float(grp["n_local"].median()),
                                  n_gene_missing=int(grp["gene_missing"].sum())))
            print(f"[{sample}] {arm:<11} d={kind}:{label:<5} "
                  f"would_detect {grp['would_detect'].mean():.4f}")
        del tx

    pred = pd.DataFrame(pred_rows)
    pred.to_csv(OUT / "detection_predicate.csv", index=False)
    display(pred)

[WT] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_WT_1/processed_data/transcripts.parquet


[WT] 103,398,068 transcripts


[WT] random floor placed 681,052/681,273 (in-tissue AND out-of-nucleus, same rule as the vicinity arms)


[WT] rejected    d=abs:5.0   would_detect 0.0376
[WT] unrejected  d=abs:5.0   would_detect 0.0429
[WT] rejected    d=abs:10.0  would_detect 0.0321
[WT] unrejected  d=abs:10.0  would_detect 0.0366
[WT] rejected    d=abs:20.0  would_detect 0.0301
[WT] unrejected  d=abs:20.0  would_detect 0.0345
[WT] rejected    d=abs:50.0  would_detect 0.0280
[WT] unrejected  d=abs:50.0  would_detect 0.0330
[WT] rejected    d=rel:2.0   would_detect 0.0628
[WT] unrejected  d=rel:2.0   would_detect 0.0659
[WT] rejected    d=rel:3.0   would_detect 0.0329
[WT] unrejected  d=rel:3.0   would_detect 0.0373


[AD] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_AD_1/processed_data/transcripts.parquet


[AD] 68,876,647 transcripts


[AD] random floor placed 398,720/398,766 (in-tissue AND out-of-nucleus, same rule as the vicinity arms)


[AD] rejected    d=abs:5.0   would_detect 0.0711
[AD] unrejected  d=abs:5.0   would_detect 0.0798
[AD] rejected    d=abs:10.0  would_detect 0.0645
[AD] unrejected  d=abs:10.0  would_detect 0.0735
[AD] rejected    d=abs:20.0  would_detect 0.0632
[AD] unrejected  d=abs:20.0  would_detect 0.0733
[AD] rejected    d=abs:50.0  would_detect 0.0592
[AD] unrejected  d=abs:50.0  would_detect 0.0691
[AD] rejected    d=rel:2.0   would_detect 0.0872
[AD] unrejected  d=rel:2.0   would_detect 0.0922
[AD] rejected    d=rel:3.0   would_detect 0.0641
[AD] unrejected  d=rel:3.0   would_detect 0.0710


,sample,arm,d_kind,d_label,n,frac_detect,median_n_local,n_gene_missing
0,WT,real,-,-,681273,0.999686,3.0,NaN
1,WT,random_tissue,-,-,681052,0.013190,0.0,NaN
2,WT,rejected,abs,5.0,680626,0.037586,0.0,0.0
3,WT,unrejected,abs,5.0,680688,0.042864,0.0,0.0
4,WT,rejected,abs,10.0,680681,0.032146,0.0,0.0
5,WT,unrejected,abs,10.0,680677,0.036643,0.0,0.0
6,WT,rejected,abs,20.0,680715,0.030137,0.0,0.0
7,WT,unrejected,abs,20.0,680754,0.034487,0.0,0.0
8,WT,rejected,abs,50.0,680893,0.028041,0.0,0.0
9,WT,unrejected,abs,50.0,680969,0.033031,0.0,0.0


## 5. The distance curve, stratified and thinned

**The decisive quantity is the gap between the real granule and any pseudo-granule, not the shape
of the pseudo curve.** If a matched sphere a few micrometres from a granule -- same region, same
density stratum, same z-plane, same radius, same seed gene -- fired nearly as often as the granule
itself, the call would be nothing but a local ambient hotspot and the reviewer would be right. It
does not: 100.0% against 4.3% at 5 um in WT, 100.0% against 8.0% in AD.

The shape of the pseudo curve is a **secondary** reading, and it supports two subordinate points:

* **The reviewer's mechanism is real, and we concede it.** At 5 um the pseudo rate sits ~3.2x (WT)
  and ~3.9x (AD) above the tissue-wide random floor, which is now built under the *same* placement
  rules as the vicinity arms (in-tissue and out-of-nucleus), so the two are comparable. Ambient
  next to a granule genuinely is enriched.
* **It is quantitatively unable to carry the conclusion.** That elevation is ~23x (WT) and ~13x
  (AD) short of the granule. The shallowness of the decay across the absolute offsets is the
  *favourable* outcome: it says the neighbourhood is nowhere near special enough to seed anything,
  so the granule's ~100% is not inherited from its surroundings.

**Read the curve on the ABSOLUTE offsets and on `rel:3.0`; `rel:2.0` is not a clean control.**
At exactly twice the source radius the copy is externally *tangent* to its source, and the
containment query carries `C.KG_BUFFER`, so a tangent copy admits a thin shell of the source
granule's own boundary transcripts -- precisely the miniball support points that define it. That
lifts `rel:2.0` above every other arm (WT 6.6%, AD 9.2%) for a reason that is geometric contact
with the source, not ambient signal. It is elevated in BOTH placement arms (WT 6.3% rejected /
6.6% unrejected), so it is not an artefact of granule-overlap rejection either.

The clean arms behave as expected and are what the argument rests on:

| | 5 um | 10 um | 20 um | 50 um | `rel:3.0` |
|---|---|---|---|---|---|
| WT | 4.3% | 3.7% | 3.4% | 3.3% | 3.7% |
| AD | 8.0% | 7.3% | 7.3% | 6.9% | 7.1% |

`rel:3.0`, whose copy clears its source entirely, sits squarely among the absolute offsets. Only
the tangent arm is anomalous, and we say why rather than quietly dropping it.

An earlier version of this notebook argued the opposite -- that `rel:2.0` sat *below* `abs:5.0`
because a tangent copy "captures none of the source's own points". That was true of a bare-radius
query and is false once the buffer is applied. The buffer is the correct choice; the reading of
this one arm had to change with it.

Two controls on the reading:

* **Stratified.** The same comparison within each local-density quintile and each brain area. A gap
  that survives within a stratum cannot be a density difference, which is the mechanism the
  reviewer proposes. Note what the strata actually show: WT is flat across quintiles (3.5-4.5%) and
  AD peaks in the **middle** quintile (11.8%) rather than the top, so there is no monotone density
  trend to report -- only a small spread, every level of which is an order of magnitude below the
  real rate.
* **Thinned.** 681K pseudo-granules drawn from 681K mutually overlapping sources are not
  independent, so a paired p-value across them would be meaningless. For anything inferential, one
  granule per 25 um spot is retained and effect sizes are reported rather than p-values.


In [7]:
if RUN_PREDICATE:
    # Consumes the `would_detect` column computed in section 4 -- refitting it here would double
    # the single most expensive step in the notebook for no new information.
    strat_rows, thin_rows = [], []
    acc_all = pseudo[pseudo["accepted"]]

    for (sample, kind, label, arm), acc in acc_all.groupby(
            ["sample", "d_kind", "d_label", "arm"], observed=True):
        for q, sub in acc.groupby("density_quintile", observed=True):
            strat_rows.append(dict(sample=sample, arm=arm, d_kind=kind, d_label=label,
                                   density_quintile=int(q), n=len(sub),
                                   frac_detect=float(sub["would_detect"].mean())))
        for area, sub in acc.groupby("brain_area", observed=True):
            strat_rows.append(dict(sample=sample, arm=arm, d_kind=kind, d_label=label,
                                   brain_area=area, n=len(sub),
                                   frac_detect=float(sub["would_detect"].mean())))

        # thinned: one per VICINITY_THIN_GRID spot, for anything inferential
        keep = (acc.assign(_gx=np.floor(acc["sphere_x"] / C.VICINITY_THIN_GRID).astype(int),
                           _gy=np.floor(acc["sphere_y"] / C.VICINITY_THIN_GRID).astype(int))
                .sample(frac=1.0, random_state=C.VICINITY_THIN_SEED)
                .drop_duplicates(["_gx", "_gy"]))
        thin_rows.append(dict(sample=sample, arm=arm, d_kind=kind, d_label=label,
                              n_thinned=len(keep),
                              frac_detect=float(keep["would_detect"].mean())))

    pd.DataFrame(strat_rows).to_csv(OUT / "detection_predicate_stratified.csv", index=False)
    pd.DataFrame(thin_rows).to_csv(OUT / "detection_predicate_thinned.csv", index=False)
    display(pd.DataFrame(thin_rows))

,sample,arm,d_kind,d_label,n_thinned,frac_detect
0,AD,rejected,abs,5.0,29332,0.031638
1,AD,unrejected,abs,5.0,29397,0.035242
2,AD,rejected,abs,10.0,29546,0.027144
3,AD,unrejected,abs,10.0,29582,0.030424
4,AD,rejected,abs,20.0,29996,0.026137
5,AD,unrejected,abs,20.0,29892,0.029874
6,AD,rejected,abs,50.0,30661,0.025276
7,AD,unrejected,abs,50.0,30629,0.028502
8,AD,rejected,rel,2.0,29293,0.062609
9,AD,unrejected,rel,2.0,29268,0.065942


## 6. The zero-placement-bias variant

The literal vicinity control has to invent sphere positions, and any placement rule is something a
reviewer can argue with. This variant has no placement rule at all.

`all_granules.parquet` is mcDETECT's **rough pass** -- every candidate aggregate, with the size,
in-soma and NC filters all switched off. It is therefore already the table of aggregates that the
pipeline saw and *rejected*, ambient-driven ones included, at real positions found by the real
detector. Comparing Set 2 against `all_granules \ Set 2` as a function of distance from each Set-2
granule asks the same question -- is nearby non-granule space different? -- with the same geometry
and the same detection machinery, and nothing synthesised.

Run **in addition** to the literal control, not instead of it: the reviewer asked for the literal
one by name.

In [8]:
if RUN_ROUGH_VARIANT:
    rough_rows = []
    for sample in C.SAMPLES:
        rough = pd.read_parquet(C.mcdetect_all_granules_path(sample))
        fine = pd.read_parquet(C.mcdetect_granules_path(sample))

        # rejected candidates = rough spheres with no fine counterpart at the same centre
        ft = cKDTree(fine[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float))
        dmin, _ = ft.query(rough[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float), k=1)
        rejected = rough[dmin > 1e-6].reset_index(drop=True)

        # distance from each rejected candidate to the nearest ACCEPTED granule
        d_to_fine, _ = ft.query(rejected[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float), k=1)
        rejected["d_to_granule"] = d_to_fine

        # Derived from the placement sweep so the two cannot drift apart; the leading 0-2 um
        # bin has no counterpart offset and is kept only to isolate near-coincident candidates.
        bins = [0, 2] + [d for d in C.VICINITY_D if d > 2] + [np.inf]
        rejected["d_bin"] = pd.cut(rejected["d_to_granule"], bins)
        agg = (rejected.groupby("d_bin", observed=True)
               .agg(n=("sphere_r", "size"), median_r=("sphere_r", "median"),
                    median_size=("size", "median"),
                    mean_in_soma=("in_soma_ratio", "mean"))
               .reset_index())
        agg["sample"] = sample
        agg["n_rough"], agg["n_fine"], agg["n_rejected"] = len(rough), len(fine), len(rejected)
        rough_rows.append(agg)
        print(f"[{sample}] rough {len(rough):,} | fine {len(fine):,} | "
              f"rejected {len(rejected):,}")

    rv = pd.concat(rough_rows, ignore_index=True)
    rv["d_bin"] = rv["d_bin"].astype(str)
    rv.to_csv(OUT / "rough_variant_by_distance.csv", index=False)
    display(rv)

[WT] rough 1,150,614 | fine 681,337 | rejected 473,351


[AD] rough 645,312 | fine 398,809 | rejected 248,790


,d_bin,n,median_r,median_size,mean_in_soma,sample,n_rough,n_fine,n_rejected
0,"(0.0, 2.0]",56627,1.071871,7.0,0.256416,WT,1150614,681337,473351
1,"(2.0, 5.0]",236645,1.005712,6.0,0.580472,WT,1150614,681337,473351
2,"(5.0, 10.0]",157426,0.955256,5.0,0.735380,WT,1150614,681337,473351
3,"(10.0, 20.0]",19364,0.930162,4.0,0.780919,WT,1150614,681337,473351
4,"(20.0, 50.0]",3138,0.918746,4.0,0.835964,WT,1150614,681337,473351
5,"(50.0, inf]",151,0.853921,3.0,0.885777,WT,1150614,681337,473351
6,"(0.0, 2.0]",23499,1.098614,7.0,0.243756,AD,645312,398809,248790
7,"(2.0, 5.0]",108382,0.999428,5.0,0.564089,AD,645312,398809,248790
8,"(5.0, 10.0]",100184,0.927878,5.0,0.762310,AD,645312,398809,248790
9,"(10.0, 20.0]",15316,0.898941,4.0,0.775459,AD,645312,398809,248790


## 7. Correctness gates

Off by default. These check the construction, not the biology -- if any fails, the numbers above
are not measuring what the section headings claim.

In [9]:
if VALIDATE:
    acc = pseudo[pseudo["accepted"]]

    # every accepted offset is on the layer_z grid
    assert set(np.round(acc["layer_z"].unique(), 6)).issubset(set(C.Z_GRID)), "off-grid layer_z"

    # every accepted offset is in tissue
    for sample in C.SAMPLES:
        mask, xb, yb = masks[sample]
        a = acc[acc["sample"] == sample]
        assert A3.in_tissue(a["sphere_x"].to_numpy(), a["sphere_y"].to_numpy(),
                            mask, xb, yb).all(), f"{sample}: offset outside tissue"

    # every accepted offset is at the intended in-plane distance from its source.
    # Paired via `src_i`, NOT the frame index: concat renumbers 0..N and the groupby pools both
    # arms, so the previous index-based pairing never matched and the assert never ran.
    n_checked = 0
    for (sample, kind, label, arm), grp in acc.groupby(
            ["sample", "d_kind", "d_label", "arm"], observed=True):
        src = sources[sample].iloc[grp["src_i"].to_numpy()]
        dd = np.hypot(grp["sphere_x"].to_numpy() - src["sphere_x"].to_numpy(),
                      grp["sphere_y"].to_numpy() - src["sphere_y"].to_numpy())
        assert np.allclose(dd, grp["offset_d"].to_numpy(), atol=1e-6), \
            f"{sample} {arm} {kind}:{label}: offset distance wrong"
        n_checked += len(grp)
    assert n_checked > 0, "the distance gate matched nothing -- it is not actually running"
    print(f"[ok] offset distance verified on {n_checked:,} pseudo-granules")

    # no accepted offset sits inside a nucleus (both arms are documented to reject this)
    for sample in C.SAMPLES:
        a = acc[acc["sample"] == sample]
        hit = np.asarray(nuc[sample].query_ball_point(
            a[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float),
            a["sphere_r"].to_numpy(float), workers=-1, return_length=True))
        assert (hit == 0).all(), f"{sample}: {int((hit>0).sum())} offsets sit inside a nucleus"
    print("[ok] no accepted offset is in a nucleus")

    # radius is matched exactly -- otherwise the count comparison is not even descriptive
    for sample in C.SAMPLES:
        src, a = sources[sample], acc[acc["sample"] == sample]
        assert set(np.round(a["sphere_r"], 9)).issubset(set(np.round(src["sphere_r"], 9)))

    # the rejected arm must actually reject: no accepted offset may overlap a real granule
    for sample in C.SAMPLES:
        a = acc[(acc["sample"] == sample) & (acc["arm"] == "rejected")]
        if len(a):
            hit, _ = A3.overlap_pairs(a, sources[sample], criterion="merge", z_col="layer_z")
            assert not hit.any(), f"{sample}: rejected arm kept an overlapping offset"

    # the real granules must clear their own predicate -- if they do not, the predicate is wrong
    if RUN_PREDICATE:
        pred = pd.read_csv(OUT / "detection_predicate.csv")
        real = pred[pred["arm"] == "real"]
        assert (real["frac_detect"] > C.PREDICATE_REAL_MIN).all(), \
            f"real granules fail their own detection predicate: {real['frac_detect'].tolist()}"

    print("[ok] all gates passed")

[ok] offset distance verified on 12,949,625 pseudo-granules


[ok] no accepted offset is in a nucleus


[ok] all gates passed


## Outputs

| file | contents |
|---|---|
| `source_summary.csv` | n source granules per sample, the seed-gene provenance, and the median NN distance of the seed-gene join |
| `placement_status.csv` | acceptance rate and mean retries per (sample, arm, offset) |
| `vicinity_overlap_with_real.csv` | fraction of unrejected offsets landing on a real granule -- a result, not a nuisance |
| `profile_summary.csv` + `profile_histogram.parquet` | real vs pseudo distributions of `n_total`, `n_marker`, in-nucleus and NC ratio |
| `profile_funnel.csv` | the same filter cascade applied to both sets, stage by stage |
| `detection_predicate.csv` | **the load-bearing table** -- fraction that would have been detected, with the real ceiling and the tissue-wide random floor |
| `detection_predicate_stratified.csv` | the same, within local-density quintile and within brain area |
| `detection_predicate_thinned.csv` | one granule per 25 um spot, for anything inferential |
| `rough_variant_by_distance.csv` | rejected rough-pass candidates by distance to the nearest accepted granule -- the no-placement-rule variant |